# 02 — The logit lens at scale: where answers surface

We run the lens over the full fact bank (one forward pass per fact) and
aggregate three statistics per site:

- the mean **answer logit** (does the lens promotion favor the answer?),
- the fraction of facts with the answer **in the top-k** (k = 5, 10, 100),
- the answer's mean **rank**.

The trajectory's last point is the model's *real* logits — a built-in sanity
check that the lens machinery is faithful.

In [ ]:
from factlens.config import DotDict
from factlens.pipelines.context import make_context
from factlens.pipelines.lens_experiment import run_lens

cfg = DotDict({
    'seed': 13, 'device': 'cpu', 'output_root': 'results',
    'model': {'name_or_path': 'gpt2', 'display_name': 'GPT-2 (124M)',
              'slug': 'gpt2', 'dtype': 'float32', 'prepend_bos': False},
    'data': {'relations': None, 'limit_per_relation': None,
             'known_only': True, 'known_rank_threshold': 5},
    'lens': {'align': 'center', 'topk': [5, 10, 100]},
    'tracking': {'tensorboard': False},
    'publish': {'figures': False},
})
# Tip: set data.limit_per_relation = 4 for a fast pass, or None for the full bank.
ctx = make_context(cfg, run_name='nb02_lens')
print(ctx.dataset.describe())

In [ ]:
out = run_lens(ctx, progress=True)
agg = out['aggregate_known']
print(f"\nknown facts: {out['aggregate_all']['known_fraction']:.0%} "
      f"({agg['n_facts']} of {out['aggregate_all']['n_facts']})")

In [ ]:
print(f"{'site':>8} | {'ans logit':>9} | {'rank':>7} | {'top-5':>6} | {'top-10':>6} | {'top-100':>7}")
print('-' * 60)
for i, site in enumerate(agg['sites']):
    print(f"{site:>8} | {agg['answer_logit_mean'][i]:>+9.2f} | "
          f"{agg['answer_rank_mean'][i]:>7.1f} | "
          f"{agg['in_top_k']['5'][i]:>6.2f} | {agg['in_top_k']['10'][i]:>6.2f} | "
          f"{agg['in_top_k']['100'][i]:>7.2f}")

In [ ]:
from factlens.analysis.figures import plot_lens_trajectory

fig = plot_lens_trajectory(agg, model_name='GPT-2 (124M)')
fig.set_constrained_layout(True)
fig

## Reading the figure

**Left panel** — the answer's logit under the lens is flat-to-negative at
early sites (the unembedding has not 'heard' about Paris yet), then rises
through the second half of the stack: this is the layer-wise construction of
the prediction that Geva et al. (2022) describe as iterative refinement.

**Right panel** — `answer in top-100` typically rises before `top-5`:
the concept enters vocabulary space diffusely first and gets *promoted*
(concentrated toward rank 1) by later sites. The dotted line marks the
boundary before `final`, the model's exact logits.

In [ ]:
# Per-fact trajectories: the lens is noisy at the single-example level.
import itertools

rows = [s for s in out['scans'] if s['known']][:8]
print(f"{'fact':<34} | " + ' | '.join(f'{s:>6}' for s in agg['sites']))
print('-' * 130)
for s in rows:
    ranks = [t['answer_rank'] for t in s['trajectory']]
    disp = ' | '.join(f'{r:>6}' if r < 10000 else f'{"inf":>6}' for r in ranks)
    print(f"{s['relation'] + ' / ' + s['subject']:<34} | " + disp)

## Known-fact coverage

'Known' here means the answer's first token ranks below 5 at the final
readout — deliberately weaker than greedy top-1 equality, because small
models often place a determiner (` the`, ` a`) above the entity without
failing to 'know' the fact. Coverage per relation is itself informative:trivia like Brasilia/Canberra is where GPT-2 gives up.

In [ ]:
from collections import defaultdict

per_rel = defaultdict(lambda: [0, 0])
for s in out['scans']:
    per_rel[s['relation']][0] += int(s['known'])
    per_rel[s['relation']][1] += 1
for rel, (k, n) in sorted(per_rel.items()):
    print(f'  {rel:<18} {k:>2}/{n:<2} known ({k / n:.0%})')

## Where this is going

The lens tells us *when the answer is readable from the vocabulary's point of
view*. It says nothing about whether the model **uses** the intermediate
representations that carry the fact — that is the job of probes (decodability)
and patches (causality), combined in notebook 03.